# Olist Product Recommendation and Customer Retention

This notebook develops CSE303 catalog topic 9.2 using the Olist Brazilian E-Commerce dataset. We will explore customer behavior, recommend relevant products, and estimate repeat-purchase likelihood. The verified pipeline will later support an application with a conversational assistant.

The proposed application user is a marketing employee exploring customer retention opportunities. The prediction horizon and application technology will be decided after assessing the data.

We will develop one section at a time and explain its actual results in [PIPELINE_NOTES.md](PIPELINE_NOTES.md). See [PROJECT_PLAN.md](PROJECT_PLAN.md) for the broader sequence.


## References

- [Olist dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) provides the source tables.
- [Mohab Allam's reference notebook](https://www.kaggle.com/code/mohabovic/eda-satisfaction-prediction-nlp-rfm-deployment/notebook) informs our approach to data preparation, EDA, satisfaction analysis, and segmentation. Specific adaptations will be credited as they are introduced.
- The course kickoff guide and Week 10 recommender starter inform the deliverables and understanding checks.

This first section contains our own loading code.


## 1 Load the source tables

Read all nine CSV files and record their dimensions. Keep the tables separate so that their row meanings and relationships can be inspected before joining them.

Open this notebook in VS Code and select a Python kernel with pandas installed. Run the cells from top to bottom with the repository root as the working directory. Source CSVs should be directly inside `data/raw/`.

`Path` constructs filesystem paths. pandas loads CSV files into DataFrames, which are tables with named columns.


In [1]:
from pathlib import Path

import pandas as pd

print(f"pandas version: {pd.__version__}")


pandas version: 3.0.1


### Locate and check the files

`data_files` maps meaningful table names to the original CSV filenames. Check that the source folder and every required file exist before loading. Relative paths let collaborators run the same code from their own repository root.


In [2]:
project_root = Path.cwd()
data_dir = project_root / "data" / "raw"

if not data_dir.is_dir():
    raise FileNotFoundError(
        "Expected data/raw/ under the working directory. "
        "Run the notebook from the repository root and check the README download instructions."
    )

data_files = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

missing_files = [
    filename
    for filename in data_files.values()
    if not (data_dir / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(f"Missing source files: {', '.join(missing_files)}")

print(f"Found all {len(data_files)} required CSV files in data/raw/.")


Found all 9 required CSV files in data/raw/.


### Read the tables and summarize their dimensions

`pd.read_csv()` reads each file into a DataFrame. The `tables` dictionary holds them under meaningful names, such as `tables["customers"]` and `tables["orders"]`.

Postal-code prefixes are identifiers. Loading them with the `string` dtype preserves leading zeros when present in the source. Other types are initially inferred by pandas. Timestamps will be inspected and parsed in a later section.

`.shape[0]` returns row count, and `.shape[1]` returns column count. These counts describe the source tables before cleaning.


In [3]:
postal_code_types = {
    "customer_zip_code_prefix": "string",
    "seller_zip_code_prefix": "string",
    "geolocation_zip_code_prefix": "string",
}

tables = {}

for table_name, filename in data_files.items():
    tables[table_name] = pd.read_csv(
        data_dir / filename,
        dtype=postal_code_types,
    )

summary_rows = []

for table_name, dataframe in tables.items():
    summary_rows.append({
        "table": table_name,
        "rows": dataframe.shape[0],
        "columns": dataframe.shape[1],
    })

data_summary = pd.DataFrame(summary_rows)

print(data_summary.to_string(index=False))
print()
print(f"Loaded {len(tables)} source tables.")


               table    rows  columns
           customers   99441        5
              orders   99441        8
         order_items  112650        7
      order_payments  103886        5
       order_reviews   99224        7
            products   32951        9
             sellers    3095        4
         geolocation 1000163        5
category_translation      71        2

Loaded 9 source tables.


### Observations from this execution

| Table | Rows | Columns |
|---|---:|---:|
| customers | 99,441 | 5 |
| orders | 99,441 | 8 |
| order_items | 112,650 | 7 |
| order_payments | 103,886 | 5 |
| order_reviews | 99,224 | 7 |
| products | 32,951 | 9 |
| sellers | 3,095 | 4 |
| geolocation | 1,000,163 | 5 |
| category_translation | 71 | 2 |

All nine source tables loaded successfully and contain rows. Their row units differ. For example, order items represent products within orders, and payments represent payment records. We will inspect keys and relationships before combining tables.

Missing-value handling, duplicate checks, timestamp parsing, and join validation will be documented in later sections.

**Next checkpoint** — Inspect sample rows, column types, missing values, identifiers, and duplicate keys.

**Suggested commit message** — `feat: start Olist notebook and load source tables`.
